In [3]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [4]:
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26.head()

,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,name
0,0.0,0.0,2025-26,1630611,Gui Santos,Gui,1610612744,GSW,Golden State Warriors,22500002,2025-10-21,GSW @ LAL,W,2.646667,0,0,0.000,0,0,0.000,0,0,0.000,0,0,0,0,1,0,0,0,0,0,0,0,-1.0,0,0,0.0,1,2:39,1,94.9,100.0,100.0,123.0,120.0,120.0,-28.0,-20.0,-20.0,0.000,0.00,0.0,0.000,0.000,0.000,100.0,100.0,0.000,0.000,0.167,0.158,101.56,99.75,83.12,99.75,-0.111,6,0.0,0.0,NaN,4.50,0.21,1,0,1,4,0,0,3,0,0,0.000,0,0,0.000,1,1,1.0,38,78,0.487,17,40,0.425,26,29,0.897,9,31,40,29,19.0,10,4,2,27,21,119,10.0,118.1,119.0,106.5,106.9,11.6,12.1,0.763,1.53,21.0,0.209,0.744,0.477,0.190,0.596,0.656,101.5,101.00,84.17,100,0.546,1610612747,LAL,Los Angeles Lakers,42,77,0.545,8,32,0.250,17,28,0.607,7,32,39,23,20.0,7,2,4,21,27,109,-10.0,106.5,106.9,118.1,119.0,-11.6,-12.1,0.548,1.15,16.9,0.256,0.791,0.523,0.196,0.597,0.610,101.5,101.00,84.17,102,0.454,0,PF,23.0,1.0,227.5,0,0.000000,0.000000,0.000000,0,1,NaN
22,19.0,20.0,2025-26,1628983,Shai Gilgeous-Alexander,Shai,1610612760,OKC,Oklahoma City Thunder,22500001,2025-10-21,OKC vs. HOU,W,47.216667,12,26,0.462,1,9,0.111,10,14,0.714,0,5,5,5,3,2,2,2,2,10,35,3,57.5,0,0,54.0,1,47:13,1,105.6,111.1,111.1,100.6,104.3,104.3,5.0,6.8,6.8,0.208,1.67,12.2,0.000,0.109,0.051,7.3,7.5,0.481,0.544,0.340,0.336,97.15,93.02,77.52,93.02,0.182,90,12.0,26.0,G,3.75,2.89,4,6,9,104,0,0,63,2,4,0.500,10,22,0.455,3,5,0.6,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12.0,12,4,5,27,26,125,1.0,107.8,112.6,103.6,107.8,4.1,4.8,0.630,2.42,18.5,0.283,0.518,0.397,0.108,0.505,0.543,97.5,93.52,77.93,111,0.521,1610612745,HOU,Houston Rockets,43,97,0.443,11,39,0.282,27,31,0.871,16,36,52,23,25.0,6,5,4,26,27,124,-1.0,103.6,107.8,107.8,112.6,-4.1,-4.8,0.535,0.92,13.9,0.482,0.717,0.603,0.217,0.500,0.560,97.5,93.52,77.93,115,0.479,0,PG,27.0,-6.5,226.0,1,0.741264,0.105895,0.105895,1,2,NaN
23,1.0,22.0,2025-26,1630559,Austin Reaves,Austin,1610612747,LAL,Los Angeles Lakers,22500002,2025-10-21,LAL vs. GSW,L,36.333333,9,16,0.563,1,5,0.200,7,10,0.700,1,4,5,9,5,2,0,1,5,10,26,-14,46.5,0,0,45.0,1,36:20,1,99.5,102.7,102.7,121.9,119.7,119.7,-22.3,-17.1,-17.1,0.409,1.80,26.5,0.031,0.148,0.085,14.7,14.5,0.594,0.637,0.301,0.308,100.43,99.74,83.12,99.74,0.147,75,9.0,16.0,F,

In [5]:
import sys
sys.path.insert(0, '.')
from src.utils import log

# base_df already loaded in your notebook
for date in ["2026-05-12", "2026-05-13", "2026-05-17", "2026-05-18"]:
    log.reconcile(date, s26)


[log] reconciled    523 props for 2026-05-12  →  53/523 hit (10.1%)
[log] reconciled    631 props for 2026-05-13  →  86/631 hit (13.6%)
[log] reconciled    592 props for 2026-05-17  →  93/592 hit (15.7%)
[log] reconciled    544 props for 2026-05-18  →  67/544 hit (12.3%)


In [10]:
# log.results_compact_view()          # all results
log.hit_rate_by("MARKET").query("MARKET in ['PTS', 'AST', 'REB']")
# log.hit_rate_by("DATE")             # hit rate by day
log.miss_breakdown()                # what types of misses
log.calibration()                   # P_OVER bucket vs actual hit rate ← this is the calibration check


,P_BUCKET,hits,total,actual_rate
0,0.10,0,1,0.000
1,0.15,2,2,1.000
2,0.20,3,9,0.333
3,0.25,2,4,0.500
4,0.30,0,14,0.000
5,0.35,3,24,0.125
6,0.40,12,46,0.261
7,0.45,14,47,0.298
8,0.50,14,39,0.359
9,0.55,15,56,0.268


In [11]:
import json, pandas as pd
df = pd.read_json('data/logs/results.jsonl', lines=True)
over = df[df['SIDE'] == 'over']
print(over.groupby('MARKET')['HIT'].agg(hits='sum', total='count', hit_rate='mean').round(3))


        hits  total  hit_rate
MARKET                       
AST        9     40     0.225
PTS       56    183     0.306
REB       29    137     0.212
